In [ ]:
# =============================================================================
# CELL 1 - COLAB ENVIRONMENT SETUP
# =============================================================================
# Installs, GPU check, Google Drive mount and Hugging Face login.
# facebook/sam3 is a GATED model: you must accept the license at
# https://huggingface.co/facebook/sam3 with the same account as your token.
# =============================================================================

!pip install -q -U transformers accelerate huggingface_hub supervision

import torch
import torchvision

print("PyTorch version    :", torch.__version__)
print("Torchvision version:", torchvision.__version__)
print("CUDA available     :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU                :", torch.cuda.get_device_name(0))
    print("GPU memory (GB)    :", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))

from google.colab import drive
drive.mount('/content/drive')

from huggingface_hub import login
login()  # paste your HF token (needs access to facebook/sam3)

print("Environment ready.")

In [ ]:
# =============================================================================
# CELL 2 - IMPORTS
# =============================================================================

import os
import gc
import csv
import time

import numpy as np
import pandas as pd
import torch

from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.lines import Line2D

import supervision as sv
from supervision.metrics import MeanAveragePrecision

from transformers import Sam3Model, Sam3Processor

# UAV images are very large; disable PIL's decompression-bomb guard.
Image.MAX_IMAGE_PIXELS = None

print("Imports OK. supervision version:", sv.__version__)

In [ ]:
# =============================================================================
# CELL 3 - CONFIGURATION
# =============================================================================
# Every parameter of the study lives here so that a thesis experiment is fully
# described by this single cell.
#
# PIPELINE (text-only prompting, NO tiling)
#   every WHOLE UAV image is downscaled so that its longest side = MAX_DIM
#   (same whole-image rule as the other no-tiling experiments),
#   the downscaled image is sent to SAM3 with ONLY a text prompt
#   (no exemplar box, nothing pasted onto the image),
#   the predicted boxes are scaled back to the original resolution,
#   overlapping duplicates are merged by NMS (offline).
#
# There are no visual prompts, so there are no anchors and no prompt plants:
# ONE run = ONE image x the text prompt, evaluated against ALL GT boxes (all_gt).
# The GT boxes are used ONLY for evaluation / visualisation, never as model input.
# =============================================================================

# ------------------------- experiment identity -------------------------------
EXPERIMENT_NAME = "text_only_no_tiling"
PROMPT_TYPE     = "text"
N_EXEMPLARS     = 0           # no visual prompts
TEXT_PROMPT     = "Rumex obtusifolius"   # short phrase (SAM3 reads at most 32 tokens)
USE_TILING      = False

# ------------------------------- dataset -------------------------------------
IMAGES_ROOT = "/content/drive/MyDrive/master_thesis/dataset/AGS_Multi_Rumex/images"
LABELS_ROOT = "/content/drive/MyDrive/master_thesis/dataset/AGS_Multi_Rumex/annotations_yolo"
RUMEX_CLASS_ID = 0
VALID_IMAGE_EXTENSIONS = (".jpg", ".jpeg", ".png")

# ------------------------------- outputs -------------------------------------
RESULTS_ROOT = f"/content/drive/MyDrive/master_thesis/results_new1/{EXPERIMENT_NAME}"

# --------------------------- whole-image input -------------------------------
MAX_DIM = 1024                # longest side of the downscaled copy fed to SAM3.
                              # An 8192 px image shrinks 8x, so a 100 px plant becomes
                              # ~12 px; the per-image table reports the median GT size
                              # at this resolution (gt_median_side_px_at_sam_input).
                              # The downscale also keeps the mask upsampling inside
                              # the SAM3 post-processing cheap.

# --------------------------- SAM3 inference ----------------------------------
# The confidence threshold is FIXED from the start. It is used both
#   (a) inside SAM3 post-processing (SAM3 runs EXACTLY ONCE per image), and
#   (b) as the operating point for precision / recall / F1 / IoU1 / IoU2.
# Because (a) == (b), AP and the operating-point metrics use the same predictions.
CONFIDENCE_THRESHOLD = 0.30
MASK_THRESHOLD = 0.40         # SAM3 mask binarisation. No plausibility filter is used,
                              # so masks do not influence any box or metric; they are
                              # discarded right after post-processing.
BATCH_SIZE = 4                # IMAGES per forward pass. Lower it if you hit CUDA OOM.
USE_FP16 = True               # half precision inference on the GPU
KEEP_MASKS = False            # masks are NEVER stored (RAM / GPU / disk / CSV / NPZ)

# ------------------------------- NMS -----------------------------------------
NMS_IOU_THRESHOLD = 0.40      # fixed from the start (no sweep, no "best" selection),
                              # applied offline (CELL 14)

# ------------------------------ evaluation ------------------------------------
EVAL_IOU_THRESHOLD = 0.50     # IoU needed for a prediction to count as a TP
EVALUATION_MODE = "all_gt"    # every GT box of the image is evaluated
                              # (held_out does not exist: there are no prompt plants)

# --------------------------- qualitative plot ---------------------------------
PLOT_MIN_GT_BOXES    = 7      # the plotted image must have at least this many GT boxes
PLOT_MAX_DISPLAY_DIM = 2048   # display-only downscale (the full image is 8192 px wide)
PLOT_SHOW_SCORES     = True   # write the confidence next to each predicted box

print(f"Configuration loaded for experiment '{EXPERIMENT_NAME}'")
print(f"  results folder : {RESULTS_ROOT}")
print(f"  prompt         : text only, '{TEXT_PROMPT}'")
print(f"  tiling         : {USE_TILING} (whole image, longest side -> {MAX_DIM} px)")
print(f"  confidence     : {CONFIDENCE_THRESHOLD} (fixed, single inference pass)")
print(f"  mask threshold : {MASK_THRESHOLD}")
print(f"  NMS IoU        : {NMS_IOU_THRESHOLD} (fixed)")
print(f"  batch / fp16   : {BATCH_SIZE} / {USE_FP16}")
print(f"  evaluation     : {EVALUATION_MODE}, IoU >= {EVAL_IOU_THRESHOLD}")
print(f"  masks stored   : {KEEP_MASKS}")

In [ ]:
# =============================================================================
# CELL 4 - OUTPUT FOLDERS
# =============================================================================
# results/<EXPERIMENT_NAME>/
#   raw_detections/      pre-NMS detections (NPZ, one file per image)
#   metrics/             per-image table, experiment summary, pooled dataset AP
#   confusion_matrices/  CSV + PNG (all_gt)
#   plots/               qualitative GT-vs-prediction figure (CELL 23)
# =============================================================================

RAW_DETECTIONS_DIR    = os.path.join(RESULTS_ROOT, "raw_detections")
METRICS_DIR           = os.path.join(RESULTS_ROOT, "metrics")
CONFUSION_MATRIX_DIR  = os.path.join(RESULTS_ROOT, "confusion_matrices")
PLOTS_DIR             = os.path.join(RESULTS_ROOT, "plots")

for d in [RESULTS_ROOT, RAW_DETECTIONS_DIR, METRICS_DIR,
          CONFUSION_MATRIX_DIR, PLOTS_DIR]:
    os.makedirs(d, exist_ok=True)

# Manifest of finished inference runs -> used for crash-safe resuming.
RUN_MANIFEST_CSV = os.path.join(RAW_DETECTIONS_DIR, "runs_manifest.csv")
MANIFEST_COLUMNS = [
    "experiment_name", "image_ID", "Prompt_Type", "text_prompt",
    "n_gt", "gt_median_side_px_at_sam_input", "n_detections_pre_nms",
    "image_width", "image_height", "sam_input_width", "sam_input_height",
    "npz_file", "inference_seconds",
]

print("Output folders ready under:", RESULTS_ROOT)

In [ ]:
# =============================================================================
# CELL 5 - SAM3 MODEL + PROCESSOR
# =============================================================================
# With USE_FP16 the weights are loaded in half precision, which roughly halves
# GPU memory and speeds up inference on a T4. Inference additionally runs inside
# torch.inference_mode() + torch.autocast (see CELL 10).
# =============================================================================

device = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_DTYPE = torch.float16 if (USE_FP16 and device == "cuda") else torch.float32

sam3_model = Sam3Model.from_pretrained(
    "facebook/sam3",
    dtype=MODEL_DTYPE,
    device_map="auto",
)
sam3_model.eval()

sam3_processor = Sam3Processor.from_pretrained("facebook/sam3")

print("SAM3 loaded.")
print("  model device:", next(sam3_model.parameters()).device)
print("  model dtype :", next(sam3_model.parameters()).dtype)

In [ ]:
# =============================================================================
# CELL 6 - DATASET AND YOLO ANNOTATION HELPERS
# =============================================================================

def load_yolo_boxes(label_path, img_width, img_height, class_id=RUMEX_CLASS_ID):
    """
    Read a YOLO .txt annotation file and convert it to pixel corner boxes.

    Input : label_path            - YOLO txt file
            img_width, img_height - size of the ORIGINAL image in pixels
            class_id              - keep only this class (0 = Rumex)
    Output: np.ndarray (N, 4) float32, boxes as [x1, y1, x2, y2] in pixels.

    YOLO stores normalised (class, x_center, y_center, width, height).
    """
    boxes = []
    with open(label_path, "r") as f:
        for line in f:
            parts = line.strip().split()
            if not parts:
                continue                      # skip empty lines
            if int(parts[0]) != class_id:
                continue                      # keep only the requested class
            xc, yc, bw, bh = map(float, parts[1:5])
            xc, yc = xc * img_width, yc * img_height
            bw, bh = bw * img_width, bh * img_height
            boxes.append([xc - bw / 2, yc - bh / 2, xc + bw / 2, yc + bh / 2])
    return np.array(boxes, dtype=np.float32).reshape(-1, 4)


def find_label_path(image_filename_no_ext, folder_name):
    """
    Locate the YOLO txt belonging to an image, supporting both a mirrored folder
    structure (labels/<folder>/<n>.txt) and a flat one (labels/<n>.txt).
    Returns the path or None.
    """
    mirrored = os.path.join(LABELS_ROOT, folder_name, image_filename_no_ext + ".txt")
    flat = os.path.join(LABELS_ROOT, image_filename_no_ext + ".txt")
    if os.path.exists(mirrored):
        return mirrored
    if os.path.exists(flat):
        return flat
    return None


def discover_images():
    """
    Walk IMAGES_ROOT and collect every image together with its label file.
    Output: list of (folder, image_path, label_path, image_id) tuples,
            where image_id = "<folder>/<image name without extension>".
    """
    records = []
    for folder in sorted(os.listdir(IMAGES_ROOT)):
        folder_path = os.path.join(IMAGES_ROOT, folder)
        if not os.path.isdir(folder_path):
            continue
        for fname in sorted(os.listdir(folder_path)):
            if not fname.lower().endswith(VALID_IMAGE_EXTENSIONS):
                continue
            name_no_ext = os.path.splitext(fname)[0]
            label_path = find_label_path(name_no_ext, folder)
            image_id = f"{folder}/{name_no_ext}"
            records.append((folder, os.path.join(folder_path, fname), label_path, image_id))
    return records


def safe_filename(image_id: str) -> str:
    """'folder/name' -> 'folder__name' so it can be used inside a file name."""
    return image_id.replace("/", "__").replace(os.sep, "__")


image_records = discover_images()
missing_labels = [r for r in image_records if r[2] is None]
valid_images = [r for r in image_records if r[2] is not None]

print(f"Discovered {len(image_records)} images in "
      f"{len(set(r[0] for r in image_records))} folders.")
print(f"With labels: {len(valid_images)} | without labels (skipped): {len(missing_labels)}")
if missing_labels:
    print("  first missing:", [r[3] for r in missing_labels[:5]])

In [ ]:
# =============================================================================
# CELL 7 - IoU
# =============================================================================

def compute_iou_matrix(boxes1, boxes2):
    """
    Pairwise IoU between two sets of [x1, y1, x2, y2] boxes.

    Input : boxes1 (N, 4), boxes2 (M, 4)
    Output: (N, M) matrix, entry [i, j] = IoU(boxes1[i], boxes2[j])
    """
    boxes1 = np.asarray(boxes1, dtype=np.float32).reshape(-1, 4)
    boxes2 = np.asarray(boxes2, dtype=np.float32).reshape(-1, 4)
    if len(boxes1) == 0 or len(boxes2) == 0:
        return np.zeros((len(boxes1), len(boxes2)), dtype=np.float32)

    x1 = np.maximum(boxes1[:, None, 0], boxes2[None, :, 0])   # left
    y1 = np.maximum(boxes1[:, None, 1], boxes2[None, :, 1])   # top
    x2 = np.minimum(boxes1[:, None, 2], boxes2[None, :, 2])   # right
    y2 = np.minimum(boxes1[:, None, 3], boxes2[None, :, 3])   # bottom

    inter = np.clip(x2 - x1, 0, None) * np.clip(y2 - y1, 0, None)
    area1 = (boxes1[:, 2] - boxes1[:, 0]) * (boxes1[:, 3] - boxes1[:, 1])
    area2 = (boxes2[:, 2] - boxes2[:, 0]) * (boxes2[:, 3] - boxes2[:, 1])
    union = area1[:, None] + area2[None, :] - inter
    return np.where(union > 0, inter / union, 0.0).astype(np.float32)


print("IoU helper ready.")

In [ ]:
# =============================================================================
# CELL 8 - CORRECT ONE-TO-ONE MATCHING
# =============================================================================
# THE RULE
#   sort predictions by confidence, high -> low
#   for each prediction:
#       look ONLY at GT boxes that are still unmatched
#       take the unmatched GT with the highest IoU
#       if that IoU >= evaluation IoU threshold -> match, else leave unmatched
#   one GT can be matched by at most one prediction, and vice versa.
#
# (The original E03 notebook took the GLOBAL best GT and discarded the
# prediction if that GT was already taken, even when another free GT had
# IoU >= threshold. That under-counted TPs on images with clustered plants.)
#
# This single function is used for TP / FP / FN / precision / recall / F1 /
# IoU1 / IoU2 / confusion matrices, so the whole thesis uses one definition.
# =============================================================================

def match_one_to_one(pred_boxes, pred_scores, gt_boxes, iou_threshold=EVAL_IOU_THRESHOLD):
    """
    Input : pred_boxes  (P, 4), pred_scores (P,), gt_boxes (G, 4), iou_threshold
    Output: dict with
        'pred_match_gt' (P,) int   - matched GT index per prediction, -1 if unmatched
        'gt_match_pred' (G,) int   - matched prediction index per GT,  -1 if unmatched
        'pred_iou'      (P,) float - IoU of the accepted match, 0.0 if unmatched
        'matched_ious'  list       - IoU values of all accepted matches
    """
    pred_boxes = np.asarray(pred_boxes, dtype=np.float32).reshape(-1, 4)
    gt_boxes = np.asarray(gt_boxes, dtype=np.float32).reshape(-1, 4)
    n_pred, n_gt = len(pred_boxes), len(gt_boxes)

    pred_match_gt = np.full(n_pred, -1, dtype=np.int64)
    gt_match_pred = np.full(n_gt, -1, dtype=np.int64)
    pred_iou = np.zeros(n_pred, dtype=np.float32)   # IoU of the accepted match per prediction

    if n_pred == 0 or n_gt == 0:
        return {"pred_match_gt": pred_match_gt, "gt_match_pred": gt_match_pred,
                "pred_iou": pred_iou, "matched_ious": []}

    iou = compute_iou_matrix(pred_boxes, gt_boxes)
    gt_free = np.ones(n_gt, dtype=bool)             # which GTs are still available

    # 'stable' keeps the original order for equal scores -> fully deterministic.
    order = np.argsort(-np.asarray(pred_scores, dtype=np.float32), kind="stable")

    matched_ious = []
    for p in order:
        if not gt_free.any():
            break                                   # every GT already has a prediction
        # Consider ONLY currently unmatched GT boxes.
        candidate_ious = np.where(gt_free, iou[p], -1.0)
        g = int(np.argmax(candidate_ious))          # best FREE GT
        if candidate_ious[g] >= iou_threshold:
            gt_free[g] = False
            pred_match_gt[p] = g
            gt_match_pred[g] = p
            pred_iou[p] = candidate_ious[g]
            matched_ious.append(float(candidate_ious[g]))

    return {"pred_match_gt": pred_match_gt, "gt_match_pred": gt_match_pred,
            "pred_iou": pred_iou, "matched_ious": matched_ious}


def safe_f1(precision, recall):
    """F1 = 2PR/(P+R) with a safe zero denominator (returns 0.0)."""
    denom = precision + recall
    return float(2.0 * precision * recall / denom) if denom > 0 else 0.0


print("One-to-one matcher ready (unmatched-GT-aware).")

In [ ]:
# =============================================================================
# CELL 9 - RESIZE HELPER (whole-image, no tiling)
# =============================================================================
# The WHOLE image is sent to SAM3 in one pass (no tiles). The large UAV image is
# downscaled so that its longest side is MAX_DIM, purely for SAM3 input size /
# speed; predictions are rescaled back to full resolution right after
# inference (see CELL 10). Same helper as the other no-tiling experiments.
# =============================================================================

def resize_for_sam3(img, max_dim=MAX_DIM):
    w, h = img.size
    scale = max_dim / max(w, h)
    if scale >= 1:
        return img, 1.0
    new_w, new_h = int(w * scale), int(h * scale)
    return img.resize((new_w, new_h), Image.BILINEAR), scale


print(f"Resize helper ready (MAX_DIM={MAX_DIM}).")

In [ ]:
# =============================================================================
# CELL 10 - SAM3 TEXT-ONLY BATCH INFERENCE  (FP16, no masks kept)
# =============================================================================
#  * BATCH_SIZE (4) downscaled WHOLE images are sent per forward pass, each with
#    the same text prompt. No exemplar box, nothing pasted onto the image.
#  * torch.inference_mode() + autocast(float16)
#  * SAM3 is called ONCE per image at CONFIDENCE_THRESHOLD = 0.30 (fixed);
#    NMS is applied later, offline (CELL 14).
#  * masks are produced by the post-processing (MASK_THRESHOLD = 0.40) but are
#    not used (no plausibility filter) -> dropped immediately.
# =============================================================================

def _to_numpy(x):
    """Torch tensor (any device/dtype) or numpy array -> float32 numpy array."""
    if torch.is_tensor(x):
        return x.detach().float().cpu().numpy()
    return np.asarray(x, dtype=np.float32)


def sam3_infer_text_batch(images_sam, text_prompt, threshold=CONFIDENCE_THRESHOLD):
    """
    Run SAM3 on a batch of downscaled whole images with a text prompt only.

    Input : images_sam  - list of downscaled PIL images (length <= BATCH_SIZE),
                          one per UAV image
            text_prompt - short phrase, e.g. "Rumex obtusifolius"
    Output: list (same length) of (boxes, scores), numpy, boxes in the
            DOWNSCALED image coordinates, every detection with score > threshold.
            Masks are NOT returned.
    """
    inputs = sam3_processor(
        images=images_sam,
        text=[text_prompt] * len(images_sam),      # one (identical) prompt per image
        return_tensors="pt",
    ).to(sam3_model.device)

    # match the pixel tensor dtype to the (possibly fp16) model weights
    if MODEL_DTYPE == torch.float16 and "pixel_values" in inputs:
        inputs["pixel_values"] = inputs["pixel_values"].to(torch.float16)

    per_image = []
    with torch.inference_mode():
        if MODEL_DTYPE == torch.float16:
            with torch.autocast("cuda", dtype=torch.float16):
                outputs = sam3_model(**inputs)
        else:
            outputs = sam3_model(**inputs)

        # target_sizes = size of each downscaled image -> boxes come back in
        # downscaled pixels (keeps the mask upsampling cheap); they are rescaled
        # to full resolution by the caller (CELL 12).
        results = sam3_processor.post_process_instance_segmentation(
            outputs,
            threshold=threshold,
            mask_threshold=MASK_THRESHOLD,
            target_sizes=inputs.get("original_sizes").tolist(),
        )

        for res in results:
            boxes = _to_numpy(res["boxes"]).reshape(-1, 4)
            scores = _to_numpy(res["scores"]).reshape(-1)
            # masks are not used (no plausibility filter) -> free them immediately
            if "masks" in res:
                res["masks"] = None
            per_image.append((boxes, scores))

    del inputs, outputs, results
    return per_image


print("SAM3 text-only batched inference helper ready (whole images, no tiling).")

In [ ]:
# =============================================================================
# CELL 11 - PRE-NMS DETECTION STORAGE
# =============================================================================
# For every run (= one image x the text prompt) we store the detections AFTER
#   SAM3 inference at CONFIDENCE_THRESHOLD -> conversion to original-image
#   coordinates
# but BEFORE NMS.
# The offline evaluation (CELL 13 onwards) therefore never needs SAM3 again.
# Masks are never stored. NPZ is compact and fast; the metric tables are CSV.
# =============================================================================

def run_npz_path(image_id):
    """Path of the NPZ holding the pre-NMS detections of one image."""
    return os.path.join(RAW_DETECTIONS_DIR, f"{safe_filename(image_id)}.npz")


def save_run_detections(image_id, boxes, scores, gt_boxes, image_size, sam_size):
    """
    Write one run's pre-NMS detections to NPZ. The file is self-contained: it also
    stores the GT boxes, so the whole offline evaluation can run without
    re-opening images or label files.
    """
    path = run_npz_path(image_id)
    np.savez_compressed(
        path,
        experiment_name=np.array(EXPERIMENT_NAME),
        image_ID=np.array(image_id),
        text_prompt=np.array(TEXT_PROMPT),
        image_width=np.array(int(image_size[0])),
        image_height=np.array(int(image_size[1])),
        sam_input_width=np.array(int(sam_size[0])),
        sam_input_height=np.array(int(sam_size[1])),
        gt_boxes=gt_boxes.astype(np.float32),
        boxes=np.asarray(boxes, dtype=np.float32).reshape(-1, 4),   # x1,y1,x2,y2 (original img)
        scores=np.asarray(scores, dtype=np.float32).reshape(-1),    # confidence > 0.30
    )
    return path


def load_run_detections(path):
    """Read one run NPZ back into a plain python dict."""
    with np.load(path, allow_pickle=False) as z:
        return {
            "image_ID": str(z["image_ID"]),
            "text_prompt": str(z["text_prompt"]),
            "image_width": int(z["image_width"]),
            "image_height": int(z["image_height"]),
            "sam_input_width": int(z["sam_input_width"]),
            "sam_input_height": int(z["sam_input_height"]),
            "gt_boxes": z["gt_boxes"].reshape(-1, 4),
            "boxes": z["boxes"].reshape(-1, 4),
            "scores": z["scores"].reshape(-1),
        }


print("Pre-NMS detection storage ready ->", RAW_DETECTIONS_DIR)

In [ ]:
# =============================================================================
# CELL 12 - MAIN GPU INFERENCE LOOP  (PHASE 1)
# =============================================================================
# FOR EACH IMAGE:
#     open the original image ONCE
#     read its GT boxes ONCE (evaluation only, never fed to SAM3)
#     downscale it ONCE to MAX_DIM and release the full-resolution copy
#     put the downscaled image into the current batch
# FOR EACH BATCH (BATCH_SIZE images):
#     run SAM3 once with the text prompt
#     map the predicted boxes back to full resolution
#     save the PRE-NMS detections (NPZ), one file per image
#
# Images without any Rumex GT box are skipped, exactly like in the exemplar
# experiments, so every experiment is evaluated on the same set of images.
# NO NMS and NO metric computation happens here - that is all done offline in
# the following cells. The loop is resumable: finished images are listed in
# runs_manifest.csv (written after every batch).
# =============================================================================

# ---- resume support ---------------------------------------------------------
done_images = set()
manifest_exists = os.path.exists(RUN_MANIFEST_CSV)
if manifest_exists:
    _m = pd.read_csv(RUN_MANIFEST_CSV)
    _m = _m[_m["experiment_name"] == EXPERIMENT_NAME]
    other_prompts = sorted(set(_m["text_prompt"].astype(str)) - {TEXT_PROMPT})
    if other_prompts:
        raise RuntimeError(
            f"{RUN_MANIFEST_CSV} already holds results for another text prompt "
            f"{other_prompts}. Use a new EXPERIMENT_NAME for '{TEXT_PROMPT}' "
            "(or delete the old results folder) so the runs are not mixed.")
    done_images = set(_m["image_ID"].astype(str))
    print(f"Resuming: {len(done_images)} images already finished for {EXPERIMENT_NAME}.")

manifest_file = open(RUN_MANIFEST_CSV, "a", newline="")
manifest_writer = csv.DictWriter(manifest_file, fieldnames=MANIFEST_COLUMNS)
if not manifest_exists:
    manifest_writer.writeheader()

start_time = time.time()
n_new_runs = 0
n_total_images = len(valid_images)
pending = []          # images waiting for the next forward pass


def process_pending_batch(batch):
    """Run SAM3 on the queued downscaled images and save their detections."""
    global n_new_runs
    t0 = time.time()
    results = sam3_infer_text_batch([p["image_sam"] for p in batch], TEXT_PROMPT)
    per_image_seconds = (time.time() - t0) / len(batch)

    for p, (boxes_sam, scores) in zip(batch, results):
        # downscaled coordinates -> full-resolution coordinates
        boxes = boxes_sam / p["scale"] if len(boxes_sam) > 0 else boxes_sam
        npz_path = save_run_detections(p["image_id"], boxes, scores, p["gt_boxes"],
                                       (p["img_w"], p["img_h"]), p["image_sam"].size)
        manifest_writer.writerow({
            "experiment_name": EXPERIMENT_NAME,
            "image_ID": p["image_id"],
            "Prompt_Type": PROMPT_TYPE,
            "text_prompt": TEXT_PROMPT,
            "n_gt": len(p["gt_boxes"]),
            "gt_median_side_px_at_sam_input": round(p["gt_median_side_sam"], 1),
            "n_detections_pre_nms": int(len(scores)),
            "image_width": p["img_w"],
            "image_height": p["img_h"],
            "sam_input_width": p["image_sam"].width,
            "sam_input_height": p["image_sam"].height,
            "npz_file": os.path.basename(npz_path),
            "inference_seconds": round(p["prep_seconds"] + per_image_seconds, 2),
        })
        n_new_runs += 1
        print(f"  [{EXPERIMENT_NAME}] ({p['img_idx']}/{n_total_images}) {p['image_id']} | "
              f"{len(p['gt_boxes'])} GT box(es), median side at SAM input "
              f"{p['gt_median_side_sam']:.1f}px | pre-NMS detections={len(scores)}")
    manifest_file.flush()                      # the whole batch is on disk -> resumable

    for p in batch:
        p["image_sam"].close()
    del results
    gc.collect()
    if device == "cuda":
        torch.cuda.empty_cache()


try:
    for img_idx, (folder, image_path, label_path, image_id) in enumerate(valid_images, start=1):
        if image_id in done_images:
            print(f"[{EXPERIMENT_NAME}] ({img_idx}/{n_total_images}) {image_id}: already done, skipped.")
            continue
        prep_t0 = time.time()

        # ---------------- open the original image exactly once ----------------
        with Image.open(image_path) as im:
            img_w, img_h = im.size
            gt_boxes = load_yolo_boxes(label_path, img_w, img_h, RUMEX_CLASS_ID)
            if len(gt_boxes) == 0:
                print(f"[{EXPERIMENT_NAME}] ({img_idx}/{n_total_images}) {image_id}: 0 GT boxes, skipped.")
                continue
            image = im.convert("RGB")

        # ---------------- downscale once, drop the full-resolution copy --------
        image_sam, scale = resize_for_sam3(image, MAX_DIM)
        if image_sam is not image:
            image.close()
        del image

        gt_side = np.maximum(gt_boxes[:, 2] - gt_boxes[:, 0], gt_boxes[:, 3] - gt_boxes[:, 1])
        pending.append({
            "img_idx": img_idx, "image_id": image_id, "gt_boxes": gt_boxes,
            "img_w": img_w, "img_h": img_h, "image_sam": image_sam, "scale": scale,
            "gt_median_side_sam": float(np.median(gt_side) * scale),
            "prep_seconds": time.time() - prep_t0,
        })

        if len(pending) == BATCH_SIZE:
            process_pending_batch(pending)
            pending = []
            elapsed = time.time() - start_time
            avg_per_image = elapsed / max(1, n_new_runs)
            eta = (n_total_images - img_idx) * avg_per_image
            print(f"[{EXPERIMENT_NAME}] {img_idx}/{n_total_images} images checked | "
                  f"{n_new_runs} new | avg/image={avg_per_image:.1f}s | "
                  f"ETA={eta / 60:.1f} min ({eta / 3600:.2f} h)")

    if pending:                                # last, incomplete batch
        process_pending_batch(pending)
        pending = []
finally:
    manifest_file.close()                      # also closed if the loop crashes

total_elapsed = time.time() - start_time
print(f"\nInference finished for {EXPERIMENT_NAME}: {n_new_runs} new images.")
print(f"Total time: {total_elapsed / 60:.1f} min ({total_elapsed / 3600:.2f} h)")
print(f"Pre-NMS detections in: {RAW_DETECTIONS_DIR}")

In [ ]:
# =============================================================================
# CELL 13 - LOAD CACHED PRE-NMS DETECTIONS  (PHASE 2 STARTS HERE)
# =============================================================================
# From here on SAM3 is never touched again. Everything below works on the NPZ
# files written in CELL 12, so you can restart the runtime, free the GPU and
# still redo the complete evaluation in a few minutes.
# To run the evaluation only: run CELLS 1-4 and 6-11 (skip CELL 5 and CELL 12),
# then continue from here.
# =============================================================================

manifest = pd.read_csv(RUN_MANIFEST_CSV)
manifest = manifest[(manifest["experiment_name"] == EXPERIMENT_NAME) &
                    (manifest["text_prompt"].astype(str) == TEXT_PROMPT)].copy()
manifest = manifest.drop_duplicates(subset=["image_ID"], keep="last")

runs = []
for _, row in manifest.iterrows():
    path = os.path.join(RAW_DETECTIONS_DIR, str(row["npz_file"]))
    if not os.path.exists(path):
        print("MISSING npz (skipped):", path)
        continue
    run = load_run_detections(path)
    run["Prompt_Type"] = str(row["Prompt_Type"])
    run["gt_median_side_px_at_sam_input"] = float(row["gt_median_side_px_at_sam_input"])
    runs.append(run)

runs_by_image = {r["image_ID"]: r for r in runs}

print(f"Loaded {len(runs)} runs (= images) for {EXPERIMENT_NAME}, prompt '{TEXT_PROMPT}'.")
print("Total pre-NMS detections:", int(sum(len(r['scores']) for r in runs)))
print("Total GT boxes:", int(sum(len(r['gt_boxes']) for r in runs)))

In [ ]:
# =============================================================================
# CELL 14 - OFFLINE NMS
# =============================================================================
# Even without tiling, SAM3 can propose more than one overlapping box for the
# same plant in a single forward pass. NMS keeps the highest-scoring box of each
# overlapping group. The NMS IoU threshold is fixed (NMS_IOU_THRESHOLD = 0.40)
# and applied offline, so the raw pre-NMS detections stay untouched on disk.
# (No tile provenance here: every detection comes from the same single input.)
# =============================================================================

def nms(boxes, scores, iou_threshold):
    """
    Input : boxes (N,4), scores (N,), iou_threshold
    Output: keep (list of kept indices, highest score first),
            n_suppressed (int)
    A detection is suppressed when its IoU with an already kept, higher-scoring
    detection is GREATER than the threshold.
    """
    n = len(boxes)
    if n == 0:
        return [], 0
    order = list(np.argsort(-np.asarray(scores, dtype=np.float32), kind="stable"))
    keep, n_suppressed = [], 0
    while order:
        i = int(order[0])
        keep.append(i)
        rest = np.array(order[1:], dtype=int)
        if rest.size == 0:
            break
        ious = compute_iou_matrix(boxes[i:i + 1], boxes[rest])[0]
        n_suppressed += int((ious > iou_threshold).sum())
        order = list(rest[ious <= iou_threshold])
    return keep, n_suppressed


def apply_nms_to_run(run, iou_threshold):
    """
    Apply NMS to one run's pre-NMS detections.

    Output: dict with 'boxes' and 'scores' of the surviving detections SORTED BY
            SCORE (high -> low), plus 'n_pre_nms' and 'n_suppressed'.
    """
    keep, n_suppressed = nms(run["boxes"], run["scores"], iou_threshold)
    keep = np.array(keep, dtype=int)
    return {
        "boxes": run["boxes"][keep].reshape(-1, 4),
        "scores": run["scores"][keep].reshape(-1),
        "n_pre_nms": int(len(run["scores"])),
        "n_suppressed": int(n_suppressed),
    }


print("Offline NMS ready. NMS IoU threshold (fixed):", NMS_IOU_THRESHOLD)

In [ ]:
# =============================================================================
# CELL 15 - EVALUATION CORE (all_gt)
# =============================================================================
# Text-only prompting uses no GT box as input, so every GT box of the image is
# evaluated (all_gt). The held_out mode of the exemplar experiments does not
# apply here: there is no prompt plant to remove.
#
#   1. match predictions to ALL GT boxes with the corrected one-to-one matcher
#      at EVAL_IOU_THRESHOLD = 0.50
#   2. matched predictions are TP, unmatched predictions are FP,
#      unmatched GT boxes are FN
# =============================================================================

STATUS_FP, STATUS_TP = 0, 1


def evaluate_predictions(pred_boxes, pred_scores, gt_boxes, eval_iou=EVAL_IOU_THRESHOLD):
    """
    Evaluate ONE prediction set against ALL GT boxes of the image.

    Output dict:
      status      (P,) int  - STATUS_TP / STATUS_FP per prediction
      TP, FP, FN, n_gt, n_pred
      precision, recall, F1
      IoU1 - mean IoU of the MATCHED prediction/GT pairs only
             ("when it finds a plant, how well is it localised?")
      IoU2 - sum of matched IoUs divided by the number of GT boxes
             ("localisation quality over ALL plants, missed ones count as 0")
    """
    pred_boxes = np.asarray(pred_boxes, dtype=np.float32).reshape(-1, 4)
    pred_scores = np.asarray(pred_scores, dtype=np.float32).reshape(-1)
    gt_boxes = np.asarray(gt_boxes, dtype=np.float32).reshape(-1, 4)

    n_pred, n_gt = len(pred_boxes), len(gt_boxes)
    status = np.full(n_pred, STATUS_FP, dtype=np.int8)   # everything FP ...

    match = match_one_to_one(pred_boxes, pred_scores, gt_boxes, eval_iou)
    status[match["pred_match_gt"] >= 0] = STATUS_TP      # ... until it is matched

    tp = int((status == STATUS_TP).sum())
    fp = int((status == STATUS_FP).sum())
    fn = int(n_gt - tp)

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / n_gt if n_gt > 0 else 0.0
    f1 = safe_f1(precision, recall)
    matched_ious = match["matched_ious"]
    iou1 = float(np.mean(matched_ious)) if matched_ious else 0.0
    iou2 = float(np.sum(matched_ious) / n_gt) if n_gt > 0 else 0.0

    return {
        "status": status, "pred_match_gt": match["pred_match_gt"],
        "gt_match_pred": match["gt_match_pred"],
        "TP": tp, "FP": fp, "FN": fn, "n_gt": n_gt, "n_pred": n_pred,
        "precision": float(precision), "recall": float(recall), "F1": float(f1),
        "IoU1": iou1, "IoU2": iou2,
    }


print("Evaluation core ready (mode:", EVALUATION_MODE, ")")

In [ ]:
# =============================================================================
# CELL 16 - AP50 AND AP50:95  (supervision.metrics.MeanAveragePrecision)
# =============================================================================
# AP is an area under the precision-recall curve, built by walking through ALL
# detections ordered by confidence. It therefore uses every post-NMS prediction
# that SAM3 returned (score > CONFIDENCE_THRESHOLD).
# Precision / recall / F1 / IoU1 / IoU2 describe ONE operating point.
# Here the operating point IS the inference threshold (0.30), so both use exactly
# the same prediction set - they still answer different questions (ranking
# quality vs. deployed behaviour).
# =============================================================================

def make_detections(boxes, scores=None):
    """NumPy boxes (+ optional scores) -> supervision Detections (class 0 = Rumex)."""
    boxes = np.asarray(boxes, dtype=np.float32).reshape(-1, 4)
    n = len(boxes)
    if scores is None:
        return sv.Detections(xyxy=boxes, class_id=np.zeros(n, dtype=int))
    return sv.Detections(xyxy=boxes,
                         confidence=np.asarray(scores, dtype=np.float32).reshape(-1),
                         class_id=np.zeros(n, dtype=int))


def compute_ap(pred_list, gt_list):
    """
    Input : two equally long lists of supervision Detections (predictions / GT).
            One entry = one evaluation episode (one run = one image).
    Output: (AP50, AP50_95). NaN when there is no GT at all to evaluate.
    Passing several episodes at once gives the POOLED (dataset-level) AP, in which
    the detections of all images are ranked together in one PR curve.
    """
    if len(gt_list) == 0 or sum(len(g) for g in gt_list) == 0:
        return float("nan"), float("nan")
    try:
        result = MeanAveragePrecision().update(pred_list, gt_list).compute()
        ap50, ap5095 = float(result.map50), float(result.map50_95)
        # supervision returns -1.0 when a metric is undefined -> report NaN instead
        return (ap50 if ap50 >= 0 else float("nan"),
                ap5095 if ap5095 >= 0 else float("nan"))
    except Exception as e:
        print("   (AP computation failed:", e, ")")
        return float("nan"), float("nan")


print("AP helpers ready (AP uses every post-NMS prediction with score > "
      f"{CONFIDENCE_THRESHOLD}).")

In [ ]:
# =============================================================================
# CELL 17 - OPERATING-POINT EVALUATION
# =============================================================================
# Keeps the predictions with score >= CONFIDENCE_THRESHOLD and evaluates them.
# The operating point is fixed from the start (no sweep, no "best" selection):
#     confidence threshold = CONFIDENCE_THRESHOLD (0.30)
#     NMS IoU threshold    = NMS_IOU_THRESHOLD    (0.40)
# =============================================================================

def evaluate_at_operating_point(nms_run, gt_boxes,
                                confidence_threshold=CONFIDENCE_THRESHOLD):
    """
    Input : nms_run  - output of apply_nms_to_run (sorted by score, high -> low)
            gt_boxes - all GT boxes of the image
    Output: (dict of evaluate_predictions, kept boxes, kept scores)
    """
    keep = nms_run["scores"] >= confidence_threshold
    boxes, scores = nms_run["boxes"][keep], nms_run["scores"][keep]
    return evaluate_predictions(boxes, scores, gt_boxes), boxes, scores


print("Operating configuration (fixed from the start, no sweep):")
print(f"  Confidence threshold = {CONFIDENCE_THRESHOLD:.2f}")
print(f"  NMS IoU threshold    = {NMS_IOU_THRESHOLD:.2f}")

In [ ]:
# =============================================================================
# CELL 18 - PER-IMAGE METRICS  (one run = one image)
# =============================================================================
# Text-only prompting has no anchors, so every image is evaluated exactly once
# and this single table replaces the run-level + image-level tables of the
# exemplar experiments. Everything is evaluated at the fixed configuration.
#   AP50 / AP50_95 : all post-NMS predictions, confidence-ranked
#   P / R / F1 / IoU1 / IoU2 / TP / FP / FN : predictions >= CONFIDENCE_THRESHOLD
# =============================================================================

METRIC_COLUMNS = ["AP50", "AP50_95", "precision", "recall", "F1", "IoU1", "IoU2"]

image_rows = []
for run in runs:
    nms_run = apply_nms_to_run(run, NMS_IOU_THRESHOLD)

    # ---- AP: every post-NMS prediction ---------------------------------------
    ap50, ap5095 = compute_ap([make_detections(nms_run["boxes"], nms_run["scores"])],
                              [make_detections(run["gt_boxes"])])

    # ---- operating point ------------------------------------------------------
    ev, _, _ = evaluate_at_operating_point(nms_run, run["gt_boxes"], CONFIDENCE_THRESHOLD)

    image_rows.append({
        "experiment_name": EXPERIMENT_NAME,
        "image_ID": run["image_ID"],
        "Prompt_Type": run["Prompt_Type"],
        "text_prompt": run["text_prompt"],
        "evaluation_mode": EVALUATION_MODE,
        "confidence_threshold": CONFIDENCE_THRESHOLD,
        "nms_iou_threshold": NMS_IOU_THRESHOLD,
        "n_gt": ev["n_gt"],
        "gt_median_side_px_at_sam_input": run["gt_median_side_px_at_sam_input"],
        "n_predictions_pre_nms": nms_run["n_pre_nms"],
        "n_suppressed_by_nms": nms_run["n_suppressed"],
        "n_predictions": ev["n_pred"],
        "AP50": ap50,
        "AP50_95": ap5095,
        "precision": ev["precision"],
        "recall": ev["recall"],
        "F1": ev["F1"],
        "IoU1": ev["IoU1"],
        "IoU2": ev["IoU2"],
        "TP": ev["TP"], "FP": ev["FP"], "FN": ev["FN"],
    })

image_level_df = pd.DataFrame(image_rows).sort_values("image_ID").reset_index(drop=True)
IMAGE_LEVEL_CSV = os.path.join(METRICS_DIR, "image_level_metrics.csv")
image_level_df.to_csv(IMAGE_LEVEL_CSV, index=False)

print(f"Per-image metrics: {len(image_level_df)} images -> {IMAGE_LEVEL_CSV}")
print(f"Images with at least one prediction: "
      f"{int((image_level_df['n_predictions'] > 0).sum())}/{len(image_level_df)}")
print(image_level_df[METRIC_COLUMNS].mean().to_string())

In [ ]:
# =============================================================================
# CELL 19 - EXPERIMENT-LEVEL SUMMARY
# =============================================================================
# Mean over images and std BETWEEN images of every per-image metric, so every
# UAV image contributes exactly the same weight regardless of how many GT boxes
# it contains.
# =============================================================================

summary_row = {
    "experiment_name": EXPERIMENT_NAME,
    "evaluation_mode": EVALUATION_MODE,
    "prompt_type": PROMPT_TYPE,
    "text_prompt": TEXT_PROMPT,
    "n_exemplars": N_EXEMPLARS,
    "use_tiling": USE_TILING,
    "max_dim": MAX_DIM,
    "confidence_threshold": CONFIDENCE_THRESHOLD,
    "nms_iou_threshold": NMS_IOU_THRESHOLD,
    "mask_threshold": MASK_THRESHOLD,
    "eval_iou_threshold": EVAL_IOU_THRESHOLD,
    "n_images": int(image_level_df["image_ID"].nunique()),
    "n_images_with_predictions": int((image_level_df["n_predictions"] > 0).sum()),
}
for col in METRIC_COLUMNS:
    summary_row[f"{col}_mean"] = image_level_df[col].mean()
    summary_row[f"{col}_std"] = image_level_df[col].std()   # spread between images

experiment_summary_df = pd.DataFrame([summary_row])
EXPERIMENT_SUMMARY_CSV = os.path.join(METRICS_DIR, "experiment_summary.csv")
experiment_summary_df.to_csv(EXPERIMENT_SUMMARY_CSV, index=False)

print(f"Experiment summary -> {EXPERIMENT_SUMMARY_CSV}\n")
print(experiment_summary_df.T.to_string(header=False))

In [ ]:
# =============================================================================
# CELL 20 - POOLED DATASET AP50 / AP50:95
# =============================================================================
# This is NOT the mean of the per-image AP values. All images are handed to
# supervision at once, so every detection of the whole dataset is ranked in ONE
# precision-recall curve. (Here one episode = one image; in the exemplar
# experiments the same image appears once per anchor.)
# =============================================================================

pred_list, gt_list = [], []
for run in runs:
    nms_run = apply_nms_to_run(run, NMS_IOU_THRESHOLD)
    pred_list.append(make_detections(nms_run["boxes"], nms_run["scores"]))
    gt_list.append(make_detections(run["gt_boxes"]))
ap50, ap5095 = compute_ap(pred_list, gt_list)   # one PR curve over ALL images
del pred_list, gt_list
gc.collect()

dataset_ap_df = pd.DataFrame([{
    "experiment_name": EXPERIMENT_NAME,
    "evaluation_mode": EVALUATION_MODE,
    "text_prompt": TEXT_PROMPT,
    "n_images": len(runs),
    "confidence_used_for_AP": CONFIDENCE_THRESHOLD,
    "nms_iou_threshold": NMS_IOU_THRESHOLD,
    "dataset_AP50": ap50,
    "dataset_AP50_95": ap5095,
}])
DATASET_AP_CSV = os.path.join(METRICS_DIR, "dataset_ap_metrics.csv")
dataset_ap_df.to_csv(DATASET_AP_CSV, index=False)

print(f"Dataset pooled AP -> {DATASET_AP_CSV}\n")
print(dataset_ap_df.to_string(index=False))
print(f"\nFor comparison, the MEAN of the per-image AP50 values "
      f"(a different quantity): {image_level_df['AP50'].mean():.4f}")

In [ ]:
# =============================================================================
# CELL 21 - DATASET-LEVEL CONFUSION MATRIX
# =============================================================================
# One class (Rumex) plus a background row/column:
#     Actual Rumex      -> Predicted Rumex      = TP
#     Actual Rumex      -> Predicted Background = FN  (missed plants)
#     Actual Background -> Predicted Rumex      = FP  (spurious detections)
#     Actual Background -> Predicted Background = not defined for detection
#                                                 (there are no true negatives)
# Counts are pooled over every image at the fixed configuration.
# =============================================================================

def plot_confusion_matrix(tp, fp, fn, title, png_path):
    """2x2 detection confusion matrix; the background/background cell stays empty."""
    matrix = np.array([[tp, fn], [fp, np.nan]], dtype=float)
    fig, ax = plt.subplots(figsize=(5.2, 4.6))
    im = ax.imshow(np.nan_to_num(matrix, nan=0.0), cmap="Blues")
    ax.set_xticks([0, 1], ["Predicted\nRumex", "Predicted\nBackground"])
    ax.set_yticks([0, 1], ["Actual\nRumex", "Actual\nBackground"])
    labels = [[f"TP\n{tp}", f"FN\n{fn}"], [f"FP\n{fp}", "n/a\n(no true\nnegatives)"]]
    vmax = np.nanmax(matrix) if np.nanmax(matrix) > 0 else 1.0
    for i in range(2):
        for j in range(2):
            value = matrix[i, j]
            colour = "white" if (not np.isnan(value) and value > 0.5 * vmax) else "black"
            ax.text(j, i, labels[i][j], ha="center", va="center",
                    color=colour, fontsize=11)
    ax.set_title(title, fontsize=11)
    fig.colorbar(im, ax=ax, fraction=0.046)
    fig.tight_layout()
    fig.savefig(png_path, dpi=200)
    plt.show()
    plt.close(fig)


tp = int(image_level_df["TP"].sum())
fp = int(image_level_df["FP"].sum())
fn = int(image_level_df["FN"].sum())
precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
f1 = safe_f1(precision, recall)

cm_df = pd.DataFrame(
    [[tp, fn], [fp, np.nan]],
    index=["actual_rumex", "actual_background"],
    columns=["predicted_rumex", "predicted_background"],
)
cm_df.to_csv(os.path.join(CONFUSION_MATRIX_DIR, f"confusion_matrix_{EVALUATION_MODE}.csv"))

plot_confusion_matrix(
    tp, fp, fn,
    f"{EXPERIMENT_NAME} - {EVALUATION_MODE}\nprompt='{TEXT_PROMPT}' | "
    f"conf={CONFIDENCE_THRESHOLD:.2f}, NMS IoU={NMS_IOU_THRESHOLD:.2f}, "
    f"eval IoU={EVAL_IOU_THRESHOLD:.2f}",
    os.path.join(CONFUSION_MATRIX_DIR, f"confusion_matrix_{EVALUATION_MODE}.png"))

confusion_summary_df = pd.DataFrame([{
    "experiment_name": EXPERIMENT_NAME, "evaluation_mode": EVALUATION_MODE,
    "TP": tp, "FP": fp, "FN": fn,
    "precision_micro": precision, "recall_micro": recall, "F1_micro": f1,
    "confidence_threshold": CONFIDENCE_THRESHOLD, "nms_iou_threshold": NMS_IOU_THRESHOLD,
    "eval_iou_threshold": EVAL_IOU_THRESHOLD,
}])
confusion_summary_df.to_csv(
    os.path.join(CONFUSION_MATRIX_DIR, "confusion_matrix_summary.csv"), index=False)

print(f"{EVALUATION_MODE}: TP={tp}  FP={fp}  FN={fn}  "
      f"P={precision:.4f}  R={recall:.4f}  F1={f1:.4f}")
print("\nConfusion matrix saved to:", CONFUSION_MATRIX_DIR)

In [ ]:
# =============================================================================
# CELL 22 - FINAL OUTPUT SUMMARY
# =============================================================================

print("=" * 78)
print(f"EXPERIMENT {EXPERIMENT_NAME} - FINAL SUMMARY")
print("=" * 78)
print(f"Prompt                   : text only, '{TEXT_PROMPT}'")
print(f"Tiling                   : {USE_TILING}  (whole image, longest side -> {MAX_DIM}px)")
print(f"Confidence threshold     : {CONFIDENCE_THRESHOLD:.2f} (fixed; SAM3 run once per image)")
print(f"Mask threshold           : {MASK_THRESHOLD:.2f}")
print(f"NMS IoU threshold        : {NMS_IOU_THRESHOLD:.2f} (fixed)")
print(f"Evaluation               : {EVALUATION_MODE}, IoU >= {EVAL_IOU_THRESHOLD:.2f}")
print(f"Images                   : {len(runs)} "
      f"({summary_row['n_images_with_predictions']} with at least one prediction)")
print("-" * 78)
print("EXPERIMENT-LEVEL RESULTS (mean over images, std between images)")
show = ["AP50_mean", "AP50_std", "AP50_95_mean", "precision_mean",
        "recall_mean", "F1_mean", "F1_std", "IoU1_mean", "IoU2_mean"]
print(experiment_summary_df[show].to_string(index=False))
print("-" * 78)
print("POOLED DATASET AP")
print(dataset_ap_df[["dataset_AP50", "dataset_AP50_95"]].to_string(index=False))
print("-" * 78)
print("POOLED CONFUSION COUNTS")
print(confusion_summary_df[["TP", "FP", "FN", "precision_micro",
                            "recall_micro", "F1_micro"]].to_string(index=False))
print("=" * 78)

print("\nFiles written under", RESULTS_ROOT)
for root, dirs, files in os.walk(RESULTS_ROOT):
    depth = root.replace(RESULTS_ROOT, "").count(os.sep)
    print("  " * depth + os.path.basename(root) + "/")
    if os.path.basename(root) == "raw_detections":
        npz_files = [f for f in files if f.endswith(".npz")]
        for f in sorted(files):
            if not f.endswith(".npz"):
                print("  " * (depth + 1) + f)
        print("  " * (depth + 1) + f"[{len(npz_files)} image NPZ files]")
    else:
        for f in sorted(files):
            print("  " * (depth + 1) + f)

In [ ]:
# =============================================================================
# CELL 23 - QUALITATIVE PLOT: BEST IMAGE, GT (left) vs PREDICTIONS (right)
# =============================================================================
# IMAGE SELECTION (per-image AP50):
#   1. keep only images that have at least PLOT_MIN_GT_BOXES (7) GT boxes
#      and take the one with the highest AP50
#   2. if NO image has 7 GT boxes: keep the images with the HIGHEST number of GT
#      boxes and take the one among them with the highest AP50
#   ties are broken by F1, then by the number of GT boxes.
#
# Right panel = post-NMS predictions of that image with
# score >= CONFIDENCE_THRESHOLD, i.e. exactly the set evaluated in CELL 18.
# The image is downscaled for DISPLAY only (PLOT_MAX_DISPLAY_DIM).
# =============================================================================

def _draw_boxes(ax, boxes, color, linewidth=1.5, linestyle="-", labels=None, fontsize=6):
    """Draw [x1,y1,x2,y2] boxes (display coordinates) on a matplotlib axis."""
    for k, (x1, y1, x2, y2) in enumerate(np.asarray(boxes).reshape(-1, 4)):
        ax.add_patch(patches.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False,
                                       edgecolor=color, linewidth=linewidth,
                                       linestyle=linestyle))
        if labels is not None:
            ax.text(x1, y1 - 2, labels[k], color="white", fontsize=fontsize,
                    va="bottom", ha="left",
                    bbox=dict(facecolor=color, edgecolor="none", pad=0.8, alpha=0.85))


def select_image_for_plot(min_gt=PLOT_MIN_GT_BOXES):
    """
    Output: (per-image row, selection_rule_text) or (None, reason)
    """
    img_df = image_level_df[image_level_df["AP50"].notna()].copy()
    if img_df.empty:
        return None, "no image with a valid AP50"

    candidates = img_df[img_df["n_gt"] >= min_gt]
    if len(candidates):
        rule = (f"highest AP50 among the {len(candidates)} images "
                f"with >= {min_gt} GT boxes")
    else:
        max_gt = int(img_df["n_gt"].max())
        candidates = img_df[img_df["n_gt"] == max_gt]
        rule = (f"no image has >= {min_gt} GT boxes -> highest AP50 among "
                f"the {len(candidates)} image(s) with the most GT boxes ({max_gt})")

    ranked = candidates.sort_values(["AP50", "F1", "n_gt"],
                                    ascending=False, na_position="last")
    print("Top candidates:")
    print(ranked[["image_ID", "n_gt", "n_predictions", "AP50", "F1",
                  "precision", "recall"]].head(5).to_string(index=False))
    return ranked.iloc[0], rule


def plot_best_image():
    row, rule = select_image_for_plot()
    if row is None:
        print("Nothing to plot:", rule)
        return
    plot_image_id = row["image_ID"]
    run = runs_by_image[plot_image_id]

    nms_run = apply_nms_to_run(run, NMS_IOU_THRESHOLD)
    keep = nms_run["scores"] >= CONFIDENCE_THRESHOLD
    pred_boxes, pred_scores = nms_run["boxes"][keep], nms_run["scores"][keep]
    gt_boxes_plot = run["gt_boxes"]

    # ---- open the image, downscale for display only ---------------------------
    image_paths = {r[3]: r[1] for r in valid_images}
    if plot_image_id not in image_paths:
        print("Image file not found for", plot_image_id)
        return
    with Image.open(image_paths[plot_image_id]) as im:
        im = im.convert("RGB")
        w, h = im.size
        s = min(1.0, PLOT_MAX_DISPLAY_DIM / max(w, h))
        disp_w, disp_h = max(1, int(w * s)), max(1, int(h * s))
        display = np.asarray(im.resize((disp_w, disp_h), Image.BILINEAR))
    to_disp = np.array([disp_w / w, disp_h / h, disp_w / w, disp_h / h], dtype=np.float32)

    # ---- figure: GT (left) | predictions (right) -------------------------------
    fig, axes = plt.subplots(1, 2, figsize=(22, 8.5))
    for ax in axes:
        ax.imshow(display)
        ax.axis("off")

    _draw_boxes(axes[0], gt_boxes_plot * to_disp, color="yellow", linewidth=1.5)
    axes[0].set_title(f"Ground truth: {len(gt_boxes_plot)} Rumex boxes", fontsize=12)

    score_labels = [f"{v:.2f}" for v in pred_scores] if PLOT_SHOW_SCORES else None
    _draw_boxes(axes[1], pred_boxes * to_disp, color="red", linewidth=1.5,
                labels=score_labels)
    axes[1].set_title(
        f"Predictions: {len(pred_boxes)} boxes | text prompt = '{run['text_prompt']}'\n"
        f"AP50={row['AP50']:.3f}  P={row['precision']:.3f}  R={row['recall']:.3f}  "
        f"F1={row['F1']:.3f}  TP={int(row['TP'])} FP={int(row['FP'])} FN={int(row['FN'])}",
        fontsize=12)

    legend_handles = [
        Line2D([0], [0], color="yellow", lw=2, label="ground truth"),
        Line2D([0], [0], color="red", lw=2, label="prediction"),
    ]
    fig.legend(handles=legend_handles, loc="lower center", ncol=2, fontsize=11,
               frameon=False)
    fig.suptitle(
        f"{EXPERIMENT_NAME} | {plot_image_id} | mode={EVALUATION_MODE}\n"
        f"no tiling (MAX_DIM={MAX_DIM}px) | conf={CONFIDENCE_THRESHOLD:.2f}, "
        f"NMS IoU={NMS_IOU_THRESHOLD:.2f}, mask thr={MASK_THRESHOLD:.2f}\n"
        f"selection: {rule}",
        fontsize=12)
    fig.tight_layout(rect=[0, 0.04, 1, 0.91])

    png_path = os.path.join(PLOTS_DIR, f"best_image_{safe_filename(plot_image_id)}_{EVALUATION_MODE}.png")
    fig.savefig(png_path, dpi=150, bbox_inches="tight")
    plt.show()
    plt.close(fig)

    print(f"\nSelected image : {plot_image_id} ({len(gt_boxes_plot)} GT boxes)")
    print(f"Selection rule : {rule}")
    print(f"Figure saved   : {png_path}")


plot_best_image()